# Movie Recommendations

This notebook uses the [MovieLens dataset](https://grouplens.org/datasets/movielens/latest/)
as well as content information that is linked through the respective movie pages on [TMDB](https://www.themoviedb.org/)

* I have included csv files under our class repo on GitHub
* License info is included in the file https://raw.githubusercontent.com/benjum/UCLAX-ML/main/Data/movielens-data/README.txt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
ratings = pd.read_csv('https://raw.githubusercontent.com/benjum/UCLAX-ML/main/Data/movielens-data/ratings.csv')
movies = pd.read_csv('https://raw.githubusercontent.com/benjum/UCLAX-ML/main/Data/movielens-data/movies.csv')

In [ ]:
ratings

In [ ]:
movies.head()

610 users and 9724 movies

In [ ]:
ratings['userId'].nunique(), ratings['movieId'].nunique()

# Idea 4: Recommend by predicting ratings with tree ensembles

The first three notebooks never trained a model in the usual supervised sense:

* **NB1 (popularity)**: rank everyone's movies by the same weighted average.  No personalization at all.
* **NB2 (collaborative filtering)**: define a similarity between users (or between movies), then take a weighted average of neighbors' ratings.
* **NB3 (content-based)**: describe each movie with text, then recommend whatever is closest in that description space.

There is another way to look at the same problem.  Every row of `ratings` is a
`(user, movie) -> rating` example.  

In [ ]:
ratings.head()

This is just a **supervised regression problem**:
The `rating` column can be treated as a target (`y`).  Given features describing a user and a movie, predict the number they would give it.

Once we can predict a rating for *any* (user, movie) pair, recommending is easy: take
the movies a user hasn't seen, predict a rating for each, and sort.

Tree ensembles (random forests, gradient boosting) are a good fit here:

* they handle counts, averages, years, and 0/1 genre flags together without any scaling
* they pick up interactions on their own (e.g. "this user rates Horror low *unless* it is very well regarded")
* they tell us afterwards which features mattered

The catch: The features have to come from `userId` and `movieId` ... but we can't feed those columns in raw.

`movieId` 318 isn't "bigger" or "better" than `movieId` 296; the numbers are just labels.
A tree that splits on `movieId < 5000` is memorizing arbitrary ID ranges.  So our job is
to replace those two ID columns with numbers that actually describe the movie and the user.

**Feature engineering is most of the
work in this notebook**; the modeling part is relatively short.

The plan:

1. Turn each `(user, movie)` pair into a row of features
2. Beware data leakage
3. Fit a decision tree, a random forest, and XGBoost, and compare them to simple baselines
4. Look at which features the model relies on
5. Make recommendations
6. Add a content feature borrowed from NB3, and see how that fares

## Features that describe the movie

In [ ]:
movies.head(2)

We'll draw two features from each record:
* the release year (inside the title string)
* the genres (a `|`-separated list)

In [ ]:
# The year is in parentheses at the end of the title
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)\s*$')[0].astype(float)
movies[['title', 'year']].head()

In [ ]:
# get_dummies with a separator turns 'Adventure|Animation|Children' into 0/1 columns
genre_dummies = movies['genres'].str.get_dummies(sep='|')
genre_dummies.head()

We'll create a genres dataframe `G` indexed by movieId, we we'll use later for a user-genre feature.

In [ ]:
G = genre_dummies.copy()
G.index = movies['movieId']

G.head()

And we can now create a table with movie features, indexed by `movieId` so it can be joined onto ratings.

In [ ]:
movie_features = pd.concat([movies[['movieId', 'year']],
                            genre_dummies.add_prefix('genre_')], axis=1).set_index('movieId')

movie_features.head()

## Split *before* engineering the rest of the features

The most useful features we can build are things like "the average rating this movie
received" and "the average rating this user gives".  But if we compute those averages
over the whole dataset, then the average for *Toy Story* already contains the test-set
rating we're about to try to predict.  Our modeling results will look artificially great but have a shortcoming for realistic application.

So: split first, compute all statistics from the training rows only, then apply
those same statistics to the test rows.

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(ratings, test_size=0.2, random_state=42)
len(train), len(test)

This random split mirrors our previous splitting and is the simplest choice, *but* the model gets to see other ratings from the same user and the same movie.  Holding out ratings *by time*, or holding out *whole users*, would be harder and more realistic tests.

## Features that describe the user and the movie's reception

These are all computed from the ratings:

| feature | meaning |
|---|---|
| `movie_count`, `movie_mean`, `movie_std` | how many ratings the movie has, and their average and spread |
| `movie_bayes` | NB1's IMDB-style weighted rating: the movie's mean, shrunk toward the global mean when it has few ratings |
| `user_count`, `user_mean`, `user_std` | how active the user is, how generous, how varied |
| `ug_affinity` | the user's average rating for movies in *this movie's genres* (also shrunk toward the global mean) |

`ug_affinity` is the only feature that combines the user *and* the movie, so it is the one that is most like a personalization characteristics.

We package this as two functions: 
* `fit_base_stats`: learn statistics from some rows
* `apply_base_stats`: turn rows into features using those statistics"

Keeping them separate lets us avoid leakage.

In [ ]:
M_SHRINK = 10   # the m from the IMDB weighted-rating formula in NB1
K_GENRE = 5     # same idea for a user's per-genre averages: shrink toward the global mean

`fit_base_stats` will learn every rating-based statistic from the rows in `fit`, and **only** those rows.

In [ ]:
def fit_base_stats(fit):
    
    C = fit['rating'].mean()

    # How is each movie received?
    movie = fit.groupby('movieId')['rating'].agg(movie_count='count', movie_mean='mean', movie_std='std')
    movie['movie_bayes'] = ((movie['movie_count'] * movie['movie_mean'] + M_SHRINK * C)
                            / (movie['movie_count'] + M_SHRINK))

    # How generous / picky is each user?
    user = fit.groupby('userId')['rating'].agg(user_count='count', user_mean='mean', user_std='std')

    # How much does each user like each genre?  (average rating they gave to movies of that genre)
    seen = G.loc[fit['movieId']].reset_index(drop=True)          # genres of each rated movie
    weighted = seen.mul(fit['rating'].to_numpy(), axis=0)        # the rating, spread over its genres
    seen['userId'] = fit['userId'].to_numpy()
    weighted['userId'] = fit['userId'].to_numpy()
    affinity = ((weighted.groupby('userId').sum() + K_GENRE * C)
                / (seen.groupby('userId').sum() + K_GENRE))

    return {'C': C, 'movie': movie, 'user': user, 'affinity': affinity}

`apply_base_stats` will turn (userId, movieId) rows into a feature dataframe using previously learned statistics.

In [ ]:
def apply_base_stats(df, stats):

    C = stats['C']
    out = (df[['userId', 'movieId']]
           .join(stats['movie'], on='movieId')
           .join(stats['user'], on='userId')
           .join(movie_features, on='movieId'))

    # Movies or users that never appeared in the fit rows fall back to the global average
    out = out.fillna({'movie_count': 0, 'movie_mean': C, 'movie_std': 0, 'movie_bayes': C,
                      'user_count': 0, 'user_mean': C, 'user_std': 0})
    out['year'] = out['year'].fillna(movie_features['year'].median())

    # Average of the user's genre affinities over the genres of this movie
    A = stats['affinity'].reindex(df['userId']).fillna(C).to_numpy()
    Gm = G.reindex(df['movieId']).fillna(0).to_numpy()
    n = Gm.sum(axis=1)
    out['ug_affinity'] = np.where(n > 0, (A * Gm).sum(axis=1) / np.maximum(n, 1), C)

    return out #.drop(columns=['userId', 'movieId'])

The test rows are easy: learn the statistics from `train`, apply them to `test`.

In [ ]:
base_stats = fit_base_stats(train)

X_test_base = apply_base_stats(test, base_stats)
X_test_base.head()

### The training rows need more care

The obvious thing for the training rows is the same recipe: `apply_base_stats(train, base_stats)`. **That is a leak.**  

Each training row's `movie_mean` and `user_mean` were computed from a set of ratings that includes *that very row*.  For a movie with only one rating, `movie_mean` *is* the target.  Test rows never get that help, so the model learns to trust these features more than it should.

The standard fix is **out-of-fold** features: split the training rows into 5 folds, and compute each fold's features from the *other four* folds.  Every training row is then described the same way a test row is: by statistics that don't contain its own rating.

`out_of_fold` will calculate features for every training row, each computed from the other folds only.


In [ ]:
from sklearn.model_selection import KFold

def out_of_fold(train, fit_fn, apply_fn, n_splits=5, seed=0):
    parts = []
    for fit_idx, hold_idx in KFold(n_splits, shuffle=True, random_state=seed).split(train):
        stats = fit_fn(train.iloc[fit_idx])
        parts.append(apply_fn(train.iloc[hold_idx], stats))
    return pd.concat(parts).loc[train.index]

In [ ]:
X_train_base = out_of_fold(train, fit_base_stats, apply_base_stats)

# we'll also keep a naive version to assess the difference
X_train_leaky = apply_base_stats(train, base_stats)     

BASE = list(X_train_base.columns)
y_train = train['rating']
y_test = test['rating']

print(len(BASE), 'features:\n', BASE)
X_train_base.shape, X_test_base.shape

## Baselines first

Never look at a model's error without something to compare it against.  Three cheap
baselines:

1. predict the global mean for everything
2. predict the movie's weighted rating [NB1's recommender]
3. predict the user's own average rating

We measure error with RMSE.

In [ ]:
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

results = {}
results['global mean'] = rmse(y_test, np.full(len(y_test), base_stats['C']))
results['movie weighted rating (NB1)'] = rmse(y_test, X_test_base['movie_bayes'])
results['user mean'] = rmse(y_test, X_test_base['user_mean'])
results

NB1's recommender, which ignores the user entirely, is *worse than just predicting each user's average rating*.  Who is rating matters more than what is being rated.

## One decision tree (to see what's happening)

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

tree = DecisionTreeRegressor(max_depth=4, random_state=42)
tree.fit(X_train_base, y_train)

results['decision tree (depth 4)'] = rmse(y_test, tree.predict(X_test_base))
results['decision tree (depth 4)']

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(tree, feature_names=BASE, max_depth=2, filled=True,
          fontsize=20, impurity=False, ax=ax)
plt.show()

This tree splits first on `ug_affinity` (does this user tend to like this movie's genres -- above or below about 3.46 stars?), then on `movie_bayes` (is the movie well regarded in general?).  Two splits in, the groups already average around 2.7 to 4.05 stars.  This tree has combined the personalized score plus NB1's recommender.

If we let the tree grow unrestricted, it would memorize the training set and generalize badly.

In [ ]:
deep_tree = DecisionTreeRegressor(random_state=42).fit(X_train_base, y_train)

print('train RMSE:', rmse(y_train, deep_tree.predict(X_train_base)))
print('test  RMSE:', rmse(y_test, deep_tree.predict(X_test_base)))

Almost zero training error, and a test error worse than predicting the global mean for everyone (1.20 vs 1.05).  The tree memorized the training set, noise included: high variance.  

We have seen how to combine trees to mitigate this: bagging and boosting.

## Random forest

**Bagging**: many trees, each grown on a bootstrap sample of the rows and allowed to consider only a random subset of features at each split.  Their errors are partly independent, so their average is far more stable than any one tree.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100,
                           min_samples_leaf=20,   # regularization hyperparameter
                           n_jobs=-1,
                           random_state=42)
rf.fit(X_train_base, y_train)

results['random forest'] = rmse(y_test, rf.predict(X_test_base))
results['random forest']

## Gradient boosting with XGBoost

**Boosting**: another way to combine trees.  Instead of averaging independent deep trees,
it grows shallow trees one after another, and each new tree is fit to the errors the
previous ones are still making.  A small `learning_rate` means each tree only corrects a
little, so you need more of them; but the result generalizes better.

[XGBoost](https://xgboost.readthedocs.io/) is the most widely used implementation, and its
`XGBRegressor` follows the usual scikit-learn `fit` / `predict` API.  Common tuning hyperparameters are:

* `n_estimators`: the maximum number of trees
* `learning_rate`: how much each tree is allowed to correct
* `max_depth`: how big each tree is
* `subsample`, `colsample_bytree`: each tree sees a random fraction of the rows / columns (the same idea as a random forest)

We also use **early stopping**: keep adding trees while the error on a validation set keeps improving, and stop when it doesn't.  The validation set is taken from the training rows - using the test set for this would risk overfitting to the test set.

In [ ]:
from xgboost import XGBRegressor

def fit_xgb(X, y):

    X_fit, X_val, y_fit, y_val = train_test_split(X, y, test_size=0.1, random_state=0)
    
    model = XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=6,
                         subsample=0.8, colsample_bytree=0.8,
                         early_stopping_rounds=50, n_jobs=-1, random_state=42)
    
    model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
    
    return model

xgb = fit_xgb(X_train_base, y_train)

results['XGBoost'] = rmse(y_test, xgb.predict(X_test_base))
print('stopped after', xgb.best_iteration + 1, 'trees')
results['XGBoost']

Once the informative features are in place, tuning these knobs moves RMSE only slightly.  The features, and how we computed them, matter far more.

### Measuring the leak

Here we look at how the performance would have changed if we did not properly handle the data leakage:  same model, same settings, same test set, but difference in how we computed feature values for the training set.

In [ ]:
xgb_leaky = fit_xgb(X_train_leaky, y_train)

pd.Series({'trained on leaky features': rmse(y_test, xgb_leaky.predict(X_test_base)),
           'trained on out-of-fold features': rmse(y_test, xgb.predict(X_test_base))})

We can get more insight by looking at how the test error varies across movies with different numbers of ratings.

In [ ]:
buckets = pd.cut(X_test_base['movie_count'], [-1, 0, 4, 19, 99, np.inf],
                 labels=['0', '1-4', '5-19', '20-99', '100+'])

pred_leaky = xgb_leaky.predict(X_test_base)
pred_oof   = xgb.predict(X_test_base)

pd.DataFrame({
    'n_test_rows': buckets.value_counts().sort_index(),
    'leaky':       [rmse(y_test[buckets == b], pred_leaky[(buckets == b).to_numpy()]) for b in buckets.cat.categories],
    'out-of-fold': [rmse(y_test[buckets == b], pred_oof[(buckets == b).to_numpy()]) for b in buckets.cat.categories],
}, index=buckets.cat.categories)

Leaky training features cost about 0.04 RMSE (0.879 vs 0.841) -- more than the gap between a random forest and XGBoost.

Large differences occurred for movies with 0 to 4 training ratings.  For a movie with a single rating, the leaky `movie_mean` is exactly the mean, so the model learns it completely; and since every leaky training row has `movie_count >= 1`, the model has never seen a movie with no ratings at all.  On the test set those are exactly the cases it faces.  For movies with 20 or more ratings, one rating among many hardly matters, and the two versions are about equal.

## Comparing the models

In [ ]:
pd.Series(results).sort_values(ascending=False).plot.barh(figsize=(8, 4))
plt.xlabel('test RMSE (lower is better)')
plt.xlim([0.7, 1.1])
plt.show()

pd.Series(results).sort_values()

Every tree model beats every baseline, and XGBoost is best.  

The progression itself is interesting too. the single depth-4 tree already captures about 60% of the improvement over the best baseline (`user mean`), and the random forest and XGBoost are close to each other. Most of the gain comes from having informative features at all.

## What did the model actually learn?

Here we look at two ways to ask "which features matter?" to a tree.

**Impurity-based importance** (scikit-learn's `feature_importances_`, also called *mean decrease in impurity*).  Every time a tree splits on a feature, the split reduces the error (the "impurity") in that node.  Add up those reductions for each feature over every split in every tree, and normalize.

* free: it's a by-product of training, no extra computation
* computed on the **training data**, so it measures how much a feature helped *fit* the training set, not how much it helps *predict* new data
* biased toward features with many distinct values (continuous columns like `year` or `user_std`), because they offer many possible split points and some will look good by chance.  A 0/1 genre flag offers only one.

**Permutation importance**.  Take the trained model and a held-out set.  Shuffle one column, which breaks its relationship to the target but keeps its distribution, and measure how much worse the predictions get.  Repeat for each column.

* measured on **held-out data**, so it reflects what the model relies on to generalize
* works for any model, not just trees
* costs extra predictions
* when two features are strongly correlated, shuffling one barely hurts because the model can lean on the other, so both can look less important than they really are

To see the difference, we add one column of pure random noise to the features and refit the random forest.  A good importance measure should give the noise column nothing.

In [ ]:
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(0)
X_train_noise = X_train_base.assign(random_noise=rng.normal(size=len(X_train_base)))
X_test_noise = X_test_base.assign(random_noise=rng.normal(size=len(X_test_base)))

rf_noise = RandomForestRegressor(n_estimators=100, min_samples_leaf=20, n_jobs=-1, random_state=42)
rf_noise.fit(X_train_noise, y_train)

# Impurity-based: comes for free with the fitted forest
impurity = pd.Series(rf_noise.feature_importances_, index=X_train_noise.columns)

# Permutation: shuffle each column of (a sample of) the TEST set, measure the increase in error
sample = X_test_noise.sample(3000, random_state=0)
perm = permutation_importance(rf_noise, sample, y_test.loc[sample.index],
                              scoring='neg_root_mean_squared_error',
                              n_repeats=5, random_state=0, n_jobs=-1)
permutation = pd.Series(perm.importances_mean, index=X_train_noise.columns)

In [ ]:
order = impurity.sort_values().index[-12:]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
impurity[order].plot.barh(ax=axes[0], color=['tab:red' if c == 'random_noise' else 'tab:blue' for c in order])
axes[0].set_title('impurity-based (training data)')
axes[0].set_xlabel('share of total impurity decrease')
permutation[order].plot.barh(ax=axes[1], color=['tab:red' if c == 'random_noise' else 'tab:blue' for c in order])
axes[1].set_title('permutation (held-out data)')
axes[1].set_xlabel('increase in test RMSE when shuffled')
plt.tight_layout()
plt.show()

In [ ]:
pd.DataFrame({'impurity rank': impurity.rank(ascending=False).astype(int),
              'permutation rank': permutation.rank(ascending=False).astype(int)}).loc[order[::-1]]

(Both panels list the features in the same order, sorted by impurity importance.)

* **They agree on the top three.**  `ug_affinity` dominates (the user's average rating for the genres of this particular movie) followed by `movie_bayes` and `movie_mean`.
* **The noise column.**  Impurity importance ranks `random_noise` 8th (of 30), with about 3% of the total, ahead of `movie_std`, `movie_count`, and every genre flag.  Permutation importance ranks it dead last: shuffling a column of noise does not hurt test predictions at all.  The forest really did split on the noise -- with 80,000 rows, some random split points reduce the training error by chance -- but those splits fit the training set, not the problem.  Impurity importance can't tell the difference; permutation importance can.
* **The same bias inflates other continuous columns.**  `year` gets about as much impurity credit as the noise, but much less on the permutation side.
* **Raw 0/1 genre flags are near the bottom on both.**  They only became useful once we combined them with the user's own history into `ug_affinity`.  The engineered interaction is the interesting part.
* **Correlated features share credit.**  `movie_bayes` and `movie_mean` carry almost the same information, so shuffling one barely hurts while the other is still there.  Permutation importance understates both. (`ug_affinity` is also partly a measure of how generous the user is, overlapping with `user_mean`.)

Rule of thumb: impurity importance is fine for a quick look; use permutation importance on held-out data before you make a claim about what matters.

XGBoost has its own `feature_importances_`.  By default it reports **gain**: the average reduction in the loss from splits on each feature.  It is the boosting version of impurity importance, computed on the training data, with the same caveats.

## Turning predictions into recommendations

Our final goal: For a given user, build a row for every movie they haven't seen,
run it through the model, and sort by predicted rating.

In [ ]:
def recommend(user_id, model, make_features, cols, n=10, min_ratings=20):
    
    seen = set(ratings.loc[ratings['userId'] == user_id, 'movieId'])

    candidates = movies.loc[~movies['movieId'].isin(seen), ['movieId', 'title', 'genres']].copy()
    candidates['userId'] = user_id

    feats = make_features(candidates)

    # Skip obscure movies: for low numbers of ratings, the movie statistics are noisy
    keep = (feats['movie_count'] >= min_ratings).to_numpy()
    candidates, feats = candidates[keep], feats[keep]

    candidates['predicted'] = model.predict(feats[cols])
    candidates['movie_mean'] = feats['movie_mean']

    return (candidates.sort_values('predicted', ascending=False)
            [['title', 'genres', 'predicted', 'movie_mean']]
            .head(n)
            .reset_index(drop=True))

def make_base_features(df):
    return apply_base_stats(df, base_stats)

In [ ]:
# Some context first: what does user 1 already love?
user_id = 1

(ratings[ratings['userId'] == user_id]
 .merge(movies[['movieId', 'title', 'genres']], on='movieId')
 .sort_values('rating', ascending=False)
 [['title', 'genres', 'rating']]
 .head(10))

In [ ]:
recommend(user_id, xgb, make_base_features, BASE)

In [ ]:
# A different user, for comparison
recommend(414, xgb, make_base_features, BASE)

The model does personalize: these two lists share only two titles.  But notice that nearly every recommendation is a well-loved classic, with an average rating of 4 or more stars (the average rating overall is about 3.5).  Sorting by predicted rating rewards consensus favorites, and personalization here is a shift of that consensus ranking by the user's genre taste.

This is a limitation.  A tree can only split on the columns we hand it, and so far `ug_affinity` is the only feature that combines the user and the movie.  We can try to add more.

## Bringing in a content feature from NB3

NB3 described each movie by its director and keywords, and recommended the movies with the most similar text.  We can turn that into a personalized feature:

* take the movies a user rated, and weight each movie's TF-IDF vector by `rating - user's average` (liked movies pull the profile toward their words, disliked movies push it away)
* add those up into one vector per user: the user's "taste profile"
* a candidate movie's feature is the cosine similarity between its vector and the user's profile

This is NB3's cosine similarity, except instead of comparing a movie to one other movie,
we compare it to everything the user has rated, weighted by how much they liked it.

With 610 users and 9,742 movies, we can compute the similarity for every user-movie pair at once as a single matrix product.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import scipy.sparse as sp

# Same text as NB3: director + keywords
movies['dir_and_keys'] = movies['director'].fillna('') + ' ' + movies['keywords'].fillna('')
X_text = TfidfVectorizer(stop_words='english', min_df=2).fit_transform(movies['dir_and_keys'])

# Position lookups: which row of the matrices belongs to which movieId / userId
mpos = pd.Series(np.arange(len(movies)), index=movies['movieId'])
upos = pd.Series(np.arange(ratings['userId'].nunique()), index=np.sort(ratings['userId'].unique()))


# Cosine similarity between every movie and 
# every user's taste profile (movies x users)
def profile_similarity(fit):

    centered = fit['rating'] - fit.groupby('userId')['rating'].transform('mean')

    R = sp.csr_matrix((centered.to_numpy(),
                       (fit['userId'].map(upos).to_numpy(), fit['movieId'].map(mpos).to_numpy())),
                      shape=(len(upos), len(mpos)))       # users x movies, centered ratings

    profiles = normalize(R @ X_text)                       # users x words, unit length
    return (X_text @ profiles.T).toarray()                 # movies x users

In [ ]:
# Sanity check on user 1: which movies are closest to their taste profile?
sim = profile_similarity(train)
pd.Series(sim[:, upos[1]], index=movies['title']).sort_values(ascending=False).head(8)

Compare with user 1's favorites in Step 11 (Se7en, The Usual Suspects, Dumb & Dumber, Billy Madison, Desperado, Pinocchio...).  It's a mixed bag: *Sword in the Stone* and *Winnie the Pooh* next to Pinocchio makes sense, and *Monty Python* sits comfortably beside the comedies, but *Night and Fog* and *Meshes of the Afternoon* are not obviously related to anything.  A profile averaged over a couple hundred ratings is blurry, and the similarities are small (the best is about 0.13).  It is a weak signal -- but weak signals can still be useful inputs to a tree.

The taste profile is built from ratings, so it gets exactly the same treatment as the features earlier: learned from `train` for the test rows, and out-of-fold for the training rows.

In [ ]:
def fit_taste_stats(fit):
    return profile_similarity(fit)

def apply_taste_stats(df, sim):
    m, u = df['movieId'].map(mpos).to_numpy(), df['userId'].map(upos).to_numpy()
    return pd.DataFrame({'taste_sim': sim[m, u]}, index=df.index)

taste_sim = fit_taste_stats(train)
X_test = pd.concat([X_test_base, apply_taste_stats(test, taste_sim)], axis=1)
X_train = pd.concat([X_train_base, out_of_fold(train, fit_taste_stats, apply_taste_stats)], axis=1)

FINAL = BASE + ['taste_sim']

In [ ]:
xgb_final = fit_xgb(X_train[FINAL], y_train)

pd.Series({'XGBoost, base features': results['XGBoost'],
           'XGBoost, + taste_sim': rmse(y_test, xgb_final.predict(X_test[FINAL]))})

In [ ]:
# Where does taste_sim help?  Same buckets as before.
pred_final = xgb_final.predict(X_test[FINAL])

pd.DataFrame({
    'base':        [rmse(y_test[buckets == b], pred_oof[(buckets == b).to_numpy()]) for b in buckets.cat.categories],
    '+ taste_sim': [rmse(y_test[buckets == b], pred_final[(buckets == b).to_numpy()]) for b in buckets.cat.categories],
}, index=buckets.cat.categories)

`taste_sim` lowers the RMSE by about a percent.  It's modest, but a larger change than tuning gave us, from a single feature.

The buckets show where the gain comes from: movies that already have plenty of ratings (100+ ratings: 0.800 to 0.779), and not movies with no ratings at all.  Content features are often pitched as a fix for movies with few ratings.  Here the feature does something else: it is a personalization signal. Among the movies everyone agrees are good, which ones fit this user?

In [ ]:
def make_final_features(df):
    return pd.concat([apply_base_stats(df, base_stats), apply_taste_stats(df, taste_sim)], axis=1)

recommend(user_id, xgb_final, make_final_features, FINAL)

In [ ]:
recommend(414, xgb_final, make_final_features, FINAL)

The lists shift: user 1 now gets *The Bridge on the River Kwai*, *Lawrence of Arabia*, *Wallace & Gromit* and *Akira*, and user 414 gets *The Shining* and *The Third Man*.  They are still mostly classics.  `taste_sim` nudges the consensus ranking rather than replace it.

## Points of Note

* Treating recommendations as supervised prediction works.  XGBoost beats NB1's popularity recommender and a per-user average by a wide margin.
* Feature engineering did most of the work. The raw genre flags were nearly useless, but the most important feature (`ug_affinity`) was one we built by combining the user's history with the movie's genres.
* Evaluation hygiene mattered more than model choice. Computing training-row statistics out-of-fold was worth about 0.04 RMSE, more than the difference between a random forest and XGBoost.
* Measure importance on held-out data: Impurity importance gave a column of pure noise more credit than half the real features; permutation importance gave it nothing.

A common production pattern combines approaches: collaborative filtering (NB2) *generates* a few hundred candidates per user, and a gradient-boosted model like this one with the CF score as an input feature *ranks* them.